# 04 — Sorguya Duyarlı Model Seçici / Router (İP-5, RQ5)

Sorgu özelliklerinden, hangi embedding modelinin bu sorguda kazanacağını tahmin eden hafif sınıflandırıcı.

**Disiplin (ihlal = sonuç geçersiz):**

- Etiketler (kazanan model) yalnızca train/val sorularının sonuçlarından üretilir.
- Test seti seçiciye hiç gösterilmez; nihai karşılaştırma bir kez, en sonda yapılır.
- En az 5 tohum; farklar eşleştirilmiş bootstrap ile raporlanır.

In [1]:
import os, sys
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin {BRANCH}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from pathlib import Path
from src.config import load_config
from src.router import sorgu_ozellikleri, ozellik_matrisi, ModelSecici

cfg = load_config()
alanlar = cfg['router']['ozellikler']
print('Router özellikleri:', alanlar)
print('Örnek:', sorgu_ozellikleri('Kayıt dondurma şartları nelerdir?', 12.4, 0.81))

# Eğer results/ altında router-veri.csv yoksa otomatik üret
router_dosyalar = sorted(Path('results').glob('ana-karsilastirma-*/router-veri.csv'))
if not router_dosyalar:
    print('router-veri.csv bulunamadı, deney_calistir.py çalıştırılıyor...')
    !python scripts/deney_calistir.py --config configs/deney.yaml
    router_dosyalar = sorted(Path('results').glob('ana-karsilastirma-*/router-veri.csv'))
son_klasor = router_dosyalar[-1].parent
print('Kullanılan deney klasörü:', son_klasor)

Cloning into '/content/RAG-proje'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 125 (delta 48), reused 96 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 152.37 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/RAG-proje
From https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri
 * branch            RAG-dev    -> FETCH_HEAD
Already up to date.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Router özellikleri: ['sorgu_uzunlugu', 'soru_tipi', 'bm25_dense_skor_farki', 'token_sayisi']
Örnek: {'sorgu_uzunlugu': 33, 'token_sayisi': 4, 'soru_tipi': 0, 'bm25_dense_skor_farki': 11.59, 'ortalama_token_uzunlugu':

## Eğitim verisi nasıl üretilir?

1. Train/val soruları tüm sistemlerde çalıştırılır (notebook 02'deki akış, `bolme` filtresi train+val).
2. Her soruda en yüksek nDCG@10 alan sistem, o sorunun etiketi olur.
3. Özellikler `sorgu_ozellikleri` ile çıkarılır; BM25–dense skor farkı için iki sistemin Top-1 skorları gerekir.
4. Sınıflandırıcı eğitilir; val üzerinde erken seçim yapılır.

In [2]:
# İP-5: Router Eğitimi (5 Tohum), Test Değerlendirmesi ve A2/A3/A6 Ablasyonları
import numpy as np
import pandas as pd
from src.data import jsonl_oku
from src.metrics import eslestirilmis_bootstrap_fark

r_df = pd.read_csv(son_klasor / 'router-veri.csv')
ozet_df = pd.read_csv(son_klasor / 'ozet.csv').set_index('sistem')
sorular_map = {s['soru_id']: s for s in jsonl_oku(cfg['veri']['sorular'])}

ndcg_piv = r_df.pivot(index='soru_id', columns='sistem', values='ndcg@10')
rec5_piv = r_df.pivot(index='soru_id', columns='sistem', values='recall@5')
top1_piv = r_df.pivot(index='soru_id', columns='sistem', values='top1_skor')

# Aday havuzu: Sözcüksel hızlı (bm25), hafif/orta dense (e5-base) ve ağır dense (bge-m3)
aday_modeller = ['bm25', 'e5-base', 'bge-m3']
gecikme_ms = {m: float(ozet_df.loc[m, 'ort_gecikme_ms']) for m in aday_modeller}

# Her soru için özellik vektörü ve maliyet-duyarlı (kalite - 0.001*ms) kazanan model etiketi
tum_ozellikler = ['sorgu_uzunlugu', 'token_sayisi', 'soru_tipi', 'bm25_dense_skor_farki', 'ortalama_token_uzunlugu']
satirlar = []
for sid, s in sorular_map.items():
    oz = sorgu_ozellikleri(
        s['soru'],
        bm25_top1=float(top1_piv.loc[sid, 'bm25']),
        dense_top1=float(top1_piv.loc[sid, 'minilm']),
    )
    # Kalite eşitse daha hızlı modeli (bm25 -> e5-base -> bge-m3) tercih et
    en_iyi = max(aday_modeller, key=lambda m: float(ndcg_piv.loc[sid, m]) - 0.0005 * gecikme_ms[m])
    oz['soru_id'] = sid
    oz['bolme'] = s['bolme']
    oz['hedef'] = en_iyi
    satirlar.append(oz)

df_oz = pd.DataFrame(satirlar).set_index('soru_id')
train_val = df_oz[df_oz['bolme'].isin(['train', 'val'])]
test_df = df_oz[df_oz['bolme'] == 'test']

def router_degerlendir(ozellik_listesi: list[str], model_turu: str = 'rf', ek_maliyet_ms: float = 0.0) -> dict:
    X_tr = train_val[ozellik_listesi].to_numpy(dtype=float)
    y_tr = train_val['hedef'].to_numpy()
    X_te = test_df[ozellik_listesi].to_numpy(dtype=float)
    tohum_ndcg, tohum_rec5, tohum_ms = [], [], []
    son_tahminler = None
    for tohum in cfg['deney']['tohumlar']:
        secici = ModelSecici(tur=model_turu, tohum=tohum)
        secici.egit(X_tr, y_tr)
        secilenler = secici.tahmin(X_te)
        son_tahminler = secilenler
        s_ndcg = [float(ndcg_piv.loc[sid, m]) for sid, m in zip(test_df.index, secilenler)]
        s_rec5 = [float(rec5_piv.loc[sid, m]) for sid, m in zip(test_df.index, secilenler)]
        s_ms = [gecikme_ms[m] + ek_maliyet_ms for m in secilenler]
        tohum_ndcg.append(s_ndcg)
        tohum_rec5.append(s_rec5)
        tohum_ms.append(np.mean(s_ms))
    ort_soru_ndcg = np.mean(tohum_ndcg, axis=0)
    return {
        'ndcg@10': round(float(np.mean(ort_soru_ndcg)), 4),
        'recall@5': round(float(np.mean(tohum_rec5)), 4),
        'ort_gecikme_ms': round(float(np.mean(tohum_ms)), 2),
        'soru_ndcg': ort_soru_ndcg,
        'secim_dagilimi': dict(pd.Series(son_tahminler).value_counts()),
    }

ana_router = router_degerlendir(tum_ozellikler, model_turu='rf')
logreg_router = router_degerlendir(tum_ozellikler, model_turu='logreg')
a2_ablasyon = router_degerlendir([f for f in tum_ozellikler if f != 'soru_tipi'], model_turu='rf')
a3_ablasyon = router_degerlendir([f for f in tum_ozellikler if f != 'ortalama_token_uzunlugu'], model_turu='rf')
a6_ablasyon = router_degerlendir(tum_ozellikler, model_turu='rf', ek_maliyet_ms=float(ozet_df.loc['minilm', 'ort_gecikme_ms']))

print('=== RQ5: Router (5 Tohum Ortalaması) ve Ablasyon Sonuçları ===')
karsilastirma = pd.DataFrame([
    {'sistem': 'Router (RF - Tam)', 'ndcg@10': ana_router['ndcg@10'], 'recall@5': ana_router['recall@5'], 'ort_gecikme_ms': ana_router['ort_gecikme_ms']},
    {'sistem': 'Router (LogReg)', 'ndcg@10': logreg_router['ndcg@10'], 'recall@5': logreg_router['recall@5'], 'ort_gecikme_ms': logreg_router['ort_gecikme_ms']},
    {'sistem': 'A2: Soru Tipi Çıkarıldı', 'ndcg@10': a2_ablasyon['ndcg@10'], 'recall@5': a2_ablasyon['recall@5'], 'ort_gecikme_ms': a2_ablasyon['ort_gecikme_ms']},
    {'sistem': 'A3: Biçimbilimsel Özellik Çıkarıldı', 'ndcg@10': a3_ablasyon['ndcg@10'], 'recall@5': a3_ablasyon['recall@5'], 'ort_gecikme_ms': a3_ablasyon['ort_gecikme_ms']},
    {'sistem': 'A6: Ön-Çıkarım Maliyeti Dâhil', 'ndcg@10': a6_ablasyon['ndcg@10'], 'recall@5': a6_ablasyon['recall@5'], 'ort_gecikme_ms': a6_ablasyon['ort_gecikme_ms']},
    {'sistem': 'Tek Model: bm25', 'ndcg@10': float(ozet_df.loc['bm25', 'ndcg@10']), 'recall@5': float(ozet_df.loc['bm25', 'recall@5']), 'ort_gecikme_ms': gecikme_ms['bm25']},
    {'sistem': 'Tek Model: e5-base', 'ndcg@10': float(ozet_df.loc['e5-base', 'ndcg@10']), 'recall@5': float(ozet_df.loc['e5-base', 'recall@5']), 'ort_gecikme_ms': gecikme_ms['e5-base']},
    {'sistem': 'Tek Model: bge-m3', 'ndcg@10': float(ozet_df.loc['bge-m3', 'ndcg@10']), 'recall@5': float(ozet_df.loc['bge-m3', 'recall@5']), 'ort_gecikme_ms': gecikme_ms['bge-m3']},
])
print(karsilastirma.to_string(index=False))
print('\nRouter model seçim dağılımı (120 test sorusu):', ana_router['secim_dagilimi'])

bm25_test_ndcg = np.array([float(ndcg_piv.loc[sid, 'bm25']) for sid in test_df.index])
e5_test_ndcg = np.array([float(ndcg_piv.loc[sid, 'e5-base']) for sid in test_df.index])
ga_bm25 = eslestirilmis_bootstrap_fark(ana_router['soru_ndcg'], bm25_test_ndcg, tekrar=1000)
ga_e5 = eslestirilmis_bootstrap_fark(ana_router['soru_ndcg'], e5_test_ndcg, tekrar=1000)
print(f'\nBootstrap (Router - BM25): fark={ga_bm25["fark"]:.4f}, GA=[{ga_bm25["alt"]:.4f}, {ga_bm25["ust"]:.4f}], anlamlı={ga_bm25["anlamli"]}')
print(f'Bootstrap (Router - e5-base): fark={ga_e5["fark"]:.4f}, GA=[{ga_e5["alt"]:.4f}, {ga_e5["ust"]:.4f}], anlamlı={ga_e5["anlamli"]}')
karsilastirma.to_csv('results/rq5-router-ozet.csv', index=False)

=== RQ5: Router (5 Tohum Ortalaması) ve Ablasyon Sonuçları ===
                             sistem  ndcg@10  recall@5  ort_gecikme_ms
                  Router (RF - Tam)   0.9018    0.9667            2.26
                    Router (LogReg)   0.8879    0.9667            1.35
            A2: Soru Tipi Çıkarıldı   0.8985    0.9750            2.24
A3: Biçimbilimsel Özellik Çıkarıldı   0.9042    0.9633            2.39
      A6: Ön-Çıkarım Maliyeti Dâhil   0.9018    0.9667           19.58
                    Tek Model: bm25   0.8479    0.9250            0.57
                 Tek Model: e5-base   0.9246    0.9750           12.33
                  Tek Model: bge-m3   0.9189    0.9833           26.07

Router model seçim dağılımı (120 test sorusu): {'bm25': np.int64(104), 'e5-base': np.int64(16)}

Bootstrap (Router - BM25): fark=0.0539, GA=[0.0221, 0.0889], anlamlı=True
Bootstrap (Router - e5-base): fark=-0.0228, GA=[-0.0628, 0.0148], anlamlı=False


## Ablasyon bağlantısı

- **A2:** Soru tipi özelliklerini çıkar → performans değişimi
- **A3:** Biçimbilimsel özellikleri çıkar → performans değişimi
- **A6:** Seçim maliyetini (ekstra çıkarım süresi) dahil et → kalite-maliyet dengesi

Router en iyi tek modeli geçemezse bu da geçerli bir bulgudur; o durumda katkı, Türkçe sorgu özelliklerinin sistematik analizi olarak raporlanır.